In [0]:
# LL notes: 
# - Overall the notebook can be simplfiied if migrated from pd to spark df
# - We recommend storing archived results in a table like {feature, importance, date} 
#   instead of adding 2 new columns at each run. It would require some refactor on the 
#   agregation code but would be more performant and maintainable.
# - the config is only being used for the run_name, evaluate its deprecation
#
# - Also: Is it needed to store this into s3 or a Volume for any reason?  

# write_local_to_s3(feature_ranks_df, f"{path}qc_feature_ranks.csv") 
#-------------------------------------------------------------------
# feature_ranks_df:pandas.core.frame.DataFrame
# feature:object
# current_rank:int64
# avg_rank:float64
# use_rate:float64

# write_local_to_s3(feature_imp_df, f"{path}qc_feature_importance.csv")
#----------------------------------------------------------------------
# feature_imp_df:pandas.core.frame.DataFrame
# feature:object
# current_importance:float64
# avg_importance:float64

In [0]:
%run ../../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append('../..')

import re
import os
import pandas as pd
import yaml
from datetime import datetime
from pyspark.sql.functions import col
from pyspark.sql import functions as F


In [0]:
config_path = "../config/config.yml"
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Missing configuration: {config_path}") 

with open(config_path, "r") as config_file:
    config = yaml.load(config_file, Loader=yaml.FullLoader)

run_name = config["shared"]["run_name"]

In [0]:
# Identifies and loads the current feature importance file.
# And loads past feature importance data from S3.

current_features_spark = (
    spark.read.format("csv").option("header", True).load(trip_feature_importance_path)
    .orderBy(col("Importance").cast("double"), ascending=False)
    .limit(10)
)

current_features = current_features_spark.toPandas()

# Rename columns to include run_name suffix
current_features = current_features.rename(
    columns={
        "Feature_Name": f"Feature_Name_{run_name}",
        "Importance": f"Importance_{run_name}"
    }
)

In [0]:
# Processes and ranks current features, formatting importance as percentages.

# Ensure the importance column is float before formatting
importance_col = f"Importance_{run_name}"
current_features[importance_col] = current_features[importance_col].astype(float)

current_features.reset_index(drop=True, inplace=True)

current_features[importance_col] = current_features[importance_col].map(lambda x: f"{round(100*x, 2)}%")

In [0]:
if spark.catalog.tableExists(trip_features_archive):
    past_features = spark.table(trip_features_archive).toPandas()
else:
    # LL Note: Old source from s3
    #---------------------------------------------------------------------------------------------------------------
    past_features_path = 's3://memberanalytics-data-out-prod/MODELDATA/QC/TRIP_PROPENSITY/featureImportance.csv'
    past_features = (
        spark.read.format("csv").option("header", True).load(past_features_path)
    ).toPandas()
    #----------------------------------------------------------------------------------------------------------------

In [0]:
# Aggregates and analyzes how often features appear and their average ranks across past runs.
# Merges current and past feature ranks.

features_vertical = pd.melt(
    past_features.filter(like="Feature")
    .reset_index()
    .rename(columns={"index": "rank"}),
    id_vars=["rank"],
    value_name="feature",
)

feature_appearances = (
    features_vertical.groupby("feature")["rank"]
    .agg(["mean", "count"])
    .reset_index()
)
feature_appearances["rate"] = (
    feature_appearances["count"]
    / past_features.filter(like="Feature").shape[1]
)
feature_appearances.sort_values(
    ["rate", "mean"], ascending=[False, True], inplace=True
)
feature_appearances.rename(
    columns={"mean": "avg_rank", "rate": "use_rate"}, inplace=True
)
feature_appearances.reset_index(drop=True, inplace=True)
feature_appearances.drop(columns=["count"], inplace=True)
feature_appearances = feature_appearances.round(2)

rename_cols = {
    col: "feature"
    for col in current_features.filter(like="Feature_Name").columns
}
rename_cols["index"] = "current_rank"
current_feature_ranks = (
    current_features.filter(like="Feature")
    .reset_index()
    .rename(columns=rename_cols)
)
current_feature_ranks = current_feature_ranks[["feature", "current_rank"]]

feature_ranks_df = pd.merge(
    current_feature_ranks, feature_appearances, how="left", on="feature"
)

#write_local_to_s3(feature_ranks_df, f"{path}qc_feature_ranks.csv")  # LL note: not needed anymore

In [0]:
# Processes and aggregates feature importances across runs.
# Loads and processes current and past model metrics (accuracy, precision, recall, ROC AUC), combining them for comparison.

feature_melt = pd.melt(
    past_features.filter(like="Feature")
    .rename(columns=lambda x: x.split("Feature_Name_")[1])
    .reset_index(),
    id_vars=["index"],
    value_name="feature",
).rename(columns={"index": "rank", "variable": "run_name"})

importance_melt = pd.melt(
    past_features.filter(like="Importance")
    .rename(columns=lambda x: x.split("Importance_")[1])
    .reset_index(),
    id_vars=["index"],
    value_name="importance",
).rename(columns={"index": "rank", "variable": "run_name"})

feature_importance_vertical = pd.merge(
    importance_melt, feature_melt, how="left", on=["rank", "run_name"]
)
feature_importance_vertical["importance"] = feature_importance_vertical[
    "importance"
].map(lambda x: float(x.rstrip("%")))

feature_importance = (
    feature_importance_vertical.groupby("feature")["importance"]
    .sum()
    .reset_index()
)
feature_importance["avg_importance"] = (
    feature_importance["importance"]
    / past_features.filter(like="Importance").shape[1]
)
feature_importance = feature_importance.round(2)
feature_importance.drop(columns=["importance"], inplace=True)
feature_importance.sort_values("avg_importance", ascending=False, inplace=True)
feature_importance.reset_index(drop=True, inplace=True)

current_importance = current_features.rename(
    columns=lambda x: x.split("_")[0].lower()
)
current_importance.rename(
    columns={"importance": "current_importance"}, inplace=True
)
current_importance["current_importance"] = current_importance[
    "current_importance"
].map(lambda x: float(x.rstrip("%")))
current_importance = current_importance[["feature", "current_importance"]]

feature_imp_df = pd.merge(
    current_importance, feature_importance, how="left", on="feature"
).fillna(0)

#write_local_to_s3(feature_imp_df, f"{path}qc_feature_importance.csv") # LL note: not needed anymore



In [0]:
# LL note:  we added a step to drop from past features any column in current feature so this process can be run several times without creating multiple columns for the same run_name
past_features_concat = pd.concat(
    [past_features.drop(columns=current_features.columns, errors='ignore'), current_features],
    axis=1
).drop_duplicates()

In [0]:
# Writes updated features and metrics back to S3 in specified locations.

past_features_df = spark.createDataFrame(past_features_concat)
past_features_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(trip_features_archive)

# write_local_to_s3(
#     past_features_concat,
#     "s3://{}/{}/{}".format(
#         config["shared"]["bucket"],
#         config["shared"]["base_path"],
#         config["shared"]["current_features"],
#     ),
# )

In [0]:
current_metric = spark.read.format("csv").option("header", True).load(trip_metrics_path).toPandas()

current_metric["iter"] = [1, 2]
current_metric = current_metric[current_metric.iter == 2]

# LL Note: This date was previously pulled from the metrics storage path but is now based on this notebook run date. 
# We assumed this notebook is target to be run after the metrics are created.

run_date = datetime.today().strftime("%Y%m%d")

current_metric["date"] = run_date

current_metric = current_metric[
    [
        "date",
        "accuracy_score",
        "precision_score",
        "recall_score",
        "roc_auc_score",
    ]
]

In [0]:
if spark.catalog.tableExists(trip_metrics_archive):
    past_metric = spark.table(trip_metrics_archive).toPandas()
else:
    # LL Note: Old source from s3 (just for debug purposes):
    #---------------------------------------------------------------------------------------------------------------
    past_metric_path = 's3://memberanalytics-data-out-prod/MODELDATA/QC/TRIP_PROPENSITY/rf_metrics.csv'
    past_metric = (
        spark.read.format("csv").option("header", True).load(past_metric_path).toPandas()
    )
    #----------------------------------------------------------------------------------------------------------------

In [0]:
past_metrics_concat = pd.concat([current_metric, past_metric]).drop_duplicates()
past_metrics_concat.reset_index(drop=True, inplace=True)

In [0]:
past_metrics_df = spark.createDataFrame(past_metrics_concat)
past_metrics_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(trip_metrics_archive)

# write_local_to_s3(
#     all_metric,
#     "s3://{}/{}/{}".format(
#         config["shared"]["bucket"],
#         config["shared"]["base_path"],
#         config["shared"]["current_metric"],
#     ),
# )